**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

**C01**

# 3D Geometry Studio
**Computer Vision**

## Lezione 8
**Quaderno di laboratorio per studenti.**

Corrispondenze, rettifica, profondità e ricostruzione sparsa. Le celle fornite consentono di eseguire tutta la baseline. Il lavoro richiesto consiste nel leggere i dati, verificare le formule, modificare una condizione e motivare una diagnosi.

**Domanda guida:** quanto della qualità della ricostruzione dipende dalla qualità dei dati?

## Dati per la consegna

Compilare prima di consegnare il notebook.


Svolgere tutti i checkpoint e i **tre esercizi aggiuntivi E01–E03**, distribuiti nei blocchi C06, C10 e C21. Scrivere le risposte negli spazi dedicati; per le verifiche numeriche aggiungere proprie celle di codice vicino all'enunciato, lasciando riconoscibili le etichette C01–C33 delle celle fornite. Conservare gli output delle prove. L'discussione conclusiva fa parte della stessa consegna individuale.

Per E02 ed E03 è fornito il codice di partenza: copiare ciascun blocco in una nuova cella e completare una sola riga. Le risposte a E01, E02 ed E03 possono comprendere i rispettivi checkpoint, senza due testi separati. Restano una sola challenge e un solo discussione conclusiva. Le quattro figure principali e il CSV sono già prodotti dalle celle fornite; non è richiesta una relazione separata.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

**C02**

### Avvio e file

**In locale.** Estrarre tutta la cartella dello ZIP studenti, aprire questo notebook con Python 3.12 dalla cartella estratta e installare le dipendenze con `%pip install -r requirements_lezione08.txt` in una nuova cella. Se l'installazione aggiorna librerie già caricate, riavviare il kernel prima di C03. Durante gli esperimenti immagini, calibrazione e modulo sono locali.

**In Google Colab.** Aprire `Geometry_3D_Studio_CV2026.ipynb` con **File > Carica notebook**. In **Runtime > Cambia tipo di runtime** scegliere Python e nessun acceleratore: la CPU è sufficiente. Il caricamento del notebook non carica automaticamente i dati. Nello stesso runtime aggiungere una cella prima di C03, copiare il blocco seguente ed eseguirlo, selezionando `Geometry_3D_Studio_studenti.zip` quando compare la scelta del file:

Per caricare il pacchetto completo e scegliere la cartella corrente seguire le istruzioni Colab nel README di questo laboratorio. Le celle scientifiche Cxx rimangono invariate.

Se Colab richiede un riavvio dopo l'installazione, riavviare la sessione e poi eseguire C03 e C05. C03 trova anche `/content/Geometry_3D_Studio_studenti`, con `geometry_studio.py` e `dati`, e stampa la cartella usata. Non cancellare il runtime dopo aver caricato i file. Una nuova sessione senza questi file richiede di nuovo caricamento, estrazione e installazione. Dopo un semplice riavvio ricostruire invece le variabili eseguendo i prerequisiti del blocco, senza confondere gli output visibili con lo stato del kernel.

**Salvataggio in Colab.** Dopo C31 scaricare il notebook eseguito dal menu **File > Scarica > Scarica .ipynb**. Per raccogliere le figure, il CSV, il JSON e il PLY della cartella `OUT`, eseguire in una nuova cella:

```python
from shutil import make_archive
from google.colab import files
files.download(make_archive("/content/risultati_lezione08", "zip", root_dir=str(OUT)))
```

Salvare anche le risposte e gli output E01–E03 nel notebook. Scaricare i risultati prima di chiudere la sessione: i file del runtime non sono conservati insieme al notebook. Le misure di riferimento della guida provengono da prove locali su CPU; non sono tempi o risultati misurati nel servizio Colab. [Istruzioni del servizio](https://research.google.com/colaboratory/faq.html).


In [ ]:
# C03
from pathlib import Path
import sys, json, time, csv
import numpy as np
import cv2
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd()/"Geometry_3D_Studio_studenti",
              Path.cwd()/"notebook", Path.cwd()/"Lezioni/08_2026-10-08_3D_Geometry_Studio/notebook"]
LAB = next((p.resolve() for p in candidates if (p/"geometry_studio.py").is_file() and (p/"dati").is_dir()), None)
assert LAB is not None, "Estrarre il pacchetto completo e aprire il notebook dalla cartella estratta."
sys.path.insert(0, str(LAB))
import geometry_studio as gs
OUT = Path.cwd()/"output_lezione08"
OUT.mkdir(exist_ok=True)
T0 = time.perf_counter()
print("Cartella laboratorio:", LAB)
print("Python", sys.version.split()[0], "OpenCV", cv2.__version__, "NumPy", np.__version__)

def mostra(fig, name=None):
    if name: fig.savefig(OUT/name, bbox_inches="tight", dpi=150)
    display(fig)
    plt.close(fig)

def tabella(rows, columns):
    def fmt(x): return f"{x:.3f}" if isinstance(x, (float, np.floating)) else str(x)
    display(Markdown("| " + " | ".join(columns) + " |\n|" + "---|"*len(columns) + "\n" +
                     "\n".join("| " + " | ".join(fmt(x) for x in r) + " |" for r in rows)))

**C04**

## Lab 0 · Il contratto dei dati
Middlebury 2014, **Adirondack-perfect**. La coppia A è già rettificata. Le immagini di lavoro sono 720×497, ottenute a 1/4 dell’originale. Le due matrici intrinseche hanno punti principali orizzontali diversi. La baseline è un dato esterno, espresso in metri.

Il riferimento di disparità serve solo alla valutazione e ai controlli della dimostrazione: non entra nella stima di F, E o dei match. L’origine completa e il ricampionamento sono documentati in `dati/README.md`.

In [ ]:
# C05
data = gs.load_data(LAB/"dati")
cal = data["calib"]
print("Forma:", data["left"].shape)
print("K sinistra:\n", cal["K0"], "\nK destra:\n", cal["K1"])
print("Baseline [m]:", cal["baseline_m"], "doffs [px]:", cal["doffs"])
assert np.isclose(cal["K1"][0,2]-cal["K0"][0,2], cal["doffs"])
mostra(gs.plot_pair(data))

**C06**

**Da completare.** Quali quantità cambiano riducendo la risoluzione di un fattore 4? Quale resta invariata?

**Risposta unica:** includere questo checkpoint nella risposta a E01 qui sotto.

---

### E01 · Ricostruire la calibrazione originale

Usare la calibrazione di C05 e `cal["S"]` (scala 0.25, traslazione −0.375).

1. Ricostruire le due matrici originali con $K_{i, \mathrm{orig}}=S^{-1}K_i$ e riportare focali, punti principali e doffs originale.
2. Per **d = 40 px nell'immagine ridotta**, calcolare disparità originale e Z nelle due griglie. Verificare che la profondità resti la stessa; indicare l'unità della baseline.
3. In 3–4 righe spiegare perché moltiplicare semplicemente i punti principali per 4 produce un errore di 1.5 px e quali quantità fisiche restano invariate.

**Una sola risposta:** calcoli e breve spiegazione coprono anche la domanda iniziale di C06. Non occorre ripetere il testo del checkpoint né creare una figura aggiuntiva.

**La mia risposta E01 e al checkpoint:**

[Inserire i calcoli o la tabella richiesti e una breve interpretazione.]

**C07**

## Lab 1 · Qualità dei match
SIFT, due vicini per descrittore, ratio test 0.75 e unicità della feature destra. Dividiamo i candidati in 80% per la stima e 20% per verifica, con seed 8. RANSAC stima F sul solo primo gruppo (soglia 1 px).

**Output:** immagine dei match, copertura 4×4, mediana e p90 della radice del residuo di Sampson. I candidati di verifica possono contenere associazioni errate: non sono una verità di riferimento. La griglia di controllo GT è distinta e non entra nel fit.

In [ ]:
# C08
match = gs.matching_lab(data)
mostra(gs.plot_matches(data, match), "01_match.png")
mostra(gs.plot_matches(data, match, grid=True))
m = match["metrics"]
tabella([["Candidati",m["candidates"]],["Per il fit",m["n_fit"]],["Verifica esclusa dal fit",m["n_heldout"]],
         ["Inlier F",m["inliers_F"]],["Inlier / fit [%]",100*m["inlier_ratio_F"]],
         ["Copertura sinistra [%]",100*m["coverage_left"]]], ["Misura", "Valore"])
mostra(gs.plot_residuals(data, match))
tabella([[k, m[k]["n"], m[k]["median"],m[k]["p90"]] for k in
         ["sampson_fit","sampson_heldout","sampson_control_gt"]], ["Popolazione", "N", "Mediana [px]", "p90 [px]"])

In [ ]:
# C09
# Controllo esplicito del denominatore e della separazione dei campioni.
inlier_ratio = int(match["inliers"].sum()) / len(match["train"])
assert set(match["train"]).isdisjoint(set(match["heldout"]))
print("Inlier ratio del fit:", round(inlier_ratio, 4))

**C10**

**Da completare.** CP1. Un alto numero di inlier e 16 celle occupate su 16 dimostrano accuratezza 3D? Che cosa aggiungono i punti esclusi dal fit?

**Risposta unica:** includere questo checkpoint nella risposta a E02 qui sotto.

---

### E02 · Soglia RANSAC e controllo fuori dal fit

Confrontare le soglie RANSAC **0.5 e 2 px** con la baseline a 1 px già calcolata in C08. Si riusano le stesse corrispondenze e la stessa suddivisione fit/verifica.

1. Copiare il codice guidato qui sotto in una nuova cella dopo C10. Completare **soltanto `indici_verifica_e02`**, scegliendo nel dizionario `match` gli indici esclusi dal fit. Il resto del codice prepara la tabella; non occorrono nuovi grafici né una ricerca della soglia ottimale.
2. Leggere insieme percentuale di inlier, copertura e mediana/p90 sui candidati esclusi dal fit. Scrivere **una conclusione di 3–5 righe**, che risponda anche a CP1: più inlier o maggiore copertura garantiscono accuratezza 3D? Che cosa aggiunge la verifica e quale limite ha?

Conservare `match` originale. Non usare GT per stimare F o per scegliere gli inlier. La maschera RANSAC riguarda soltanto il fit; i candidati di verifica possono contenere abbinamenti sbagliati.

**Codice di partenza — una riga da completare:**

```python
# E02 — una sola riga da completare; C03, C05 e C08 già eseguite
p0_e02, p1_e02 = match["p0"], match["p1"]
train_e02 = match["train"]
indici_verifica_e02 = ...  # TODO: indici dei candidati esclusi dal fit
if indici_verifica_e02 is Ellipsis:
    raise ValueError("Completare indici_verifica_e02 prima di eseguire.")
indici_verifica_e02 = np.asarray(indici_verifica_e02, dtype=int)
assert len(indici_verifica_e02) > 0
assert not set(train_e02) & set(indici_verifica_e02)

# La soglia 1 px è la baseline già calcolata, non si ricalcolano i match.
modelli_e02 = {1.0: (match["F"], match["inliers"][train_e02])}
for soglia_e02 in (0.5, 2.0):
    cv2.setRNGSeed(8)
    F_e02, maschera_e02 = cv2.findFundamentalMat(
        p0_e02[train_e02], p1_e02[train_e02], cv2.FM_RANSAC,
        soglia_e02, 0.999, 10000)
    if F_e02 is None or F_e02.shape != (3, 3) or maschera_e02 is None:
        print("F non valutabile per soglia", soglia_e02)
        continue
    modelli_e02[soglia_e02] = (F_e02, maschera_e02.ravel().astype(bool))

righe_e02 = []
for soglia_e02 in sorted(modelli_e02):
    F_e02, maschera_e02 = modelli_e02[soglia_e02]
    accettati_e02 = train_e02[maschera_e02]
    residui_e02 = gs.sampson_px(
        F_e02, p0_e02[indici_verifica_e02], p1_e02[indici_verifica_e02])
    q_e02 = gs.quantiles(residui_e02)
    righe_e02.append([
        soglia_e02, len(accettati_e02), len(train_e02),
        100*len(accettati_e02)/len(train_e02),
        100*gs.coverage(p0_e02[accettati_e02], data["left"].shape),
        100*gs.coverage(p1_e02[accettati_e02], data["right"].shape),
        q_e02["n"], q_e02["median"], q_e02["p90"]])
tabella(righe_e02, ["Soglia px", "Inlier", "Fit N", "Inlier %", "Cop. L %",
                     "Cop. R %", "Verifica N", "Mediana px", "p90 px"])
```

**La mia risposta E02 e al checkpoint:**

[Inserire i calcoli o la tabella richiesti e una breve interpretazione.]

**C11**

## Lab 2 · Rettifica misurabile
La coppia B nasce da A mediante rotazioni virtuali note di +3° e −3° attorno agli assi ottici. Non rappresenta una nuova acquisizione. Le trasformazioni aggiornano anche la posa relativa. `stereoRectify` e `remap` riportano le corrispondenze su righe comuni.

**Output:** prima/dopo, maschere dei pixel campionati e residuo verticale sugli stessi controlli visibili. I controlli trasformati analiticamente verificano la coerenza delle matrici: un residuo quasi nullo qui non misura la precisione di un sensore reale.

In [ ]:
# C12
rect = gs.rectification_lab(data)
mostra(gs.plot_rectification(rect), "02_rettifica.png")
mostra(gs.plot_rectification_masks(rect))
r = rect["metrics"]
tabella([["Prima",r["before"]["n"],r["before"]["median"],r["before"]["p90"]],
         ["Dopo",r["after"]["n"],r["after"]["median"],r["after"]["p90"]]],
        ["Controlli analitici", "N", "Mediana verticale [px]", "p90 [px]"])
print("Residuo massimo dopo [px]:", rect["after_residual"].max())
print("Frazione pixel campionati L/R:", r["valid_area_left"],r["valid_area_right"])
assert len(rect["before_residual"]) == len(rect["after_residual"])
assert rect["after_residual"].max() < 1e-7

In [ ]:
# C13
# Un secondo controllo usa nuove feature sulle immagini ricampionate.
# Non si selezionano i match in base al loro errore verticale.
a0,a1,info = gs.match_images(rect["rect0"],rect["rect1"])
rv_osservato = np.abs(a0[:,1]-a1[:,1])
print("Nuovi candidati dopo ricampionamento:", gs.quantiles(rv_osservato))
# Questa popolazione contiene rumore di localizzazione e possibili falsi match.

**C14**

**Da completare.** CP2. Perché il test sulle coordinate dà circa zero mentre i nuovi match possono avere residui non nulli? Che significato hanno le zone nere?

*La mia risposta:* …

**C15**

### Salvataggio e controllo dello stato
Salvare il notebook. Alla ripresa si torna alla **coppia A originale**. La matrice Q generata per la coppia B rettificata appartiene a quel sistema di coordinate e non va applicata alla disparità di A.

**C16**

## Lab 3 · Disparità, profondità e riferimento
Osservare tre finestre e il costo locale lungo la stessa scanline. Il minimo del costo SAD aiuta a capire l’ambiguità locale. StereoSGBM aggrega costi con regolarizzazione: le curve SAD sono una dimostrazione separata, non il costo interno esatto di OpenCV.

Le immagini in alto mostrano il contesto di 51×51 pixel. Il riquadro giallo delimita la finestra centrale di 9×9 pixel effettivamente usata per la SAD; la croce indica il pixel di riferimento. Le colonne A, B e C associano ogni finestra alla curva sottostante. La deviazione standard è calcolata sui livelli di grigio del solo riquadro 9×9, e tutti i grafici usano le stesse scale. Il punto colorato indica il minimo locale della SAD; la linea tratteggiata indica la disparità di riferimento GT.

Parametri baseline: 80 disparità da 0, blocco 5, P1=200, P2=800, unicità 10. Il campo di ricerca è fissato prima della valutazione. Non si sceglie la configurazione ottimizzando questa GT.

In [ ]:
# C17
mostra(gs.plot_scanline(data))
dense = gs.stereo_lab(data)
mostra(gs.plot_dense(data, dense), "03_disparita_profondita.png")
mostra(gs.plot_reference(data, dense))
gs.show_metrics(dense["metrics"])

**C18**

### La formula nel sistema della coppia A
Con $d=u_L-u_R$ e $\mathrm{doffs}=c_{xR}-c_{xL}$:

$$Z=\frac{fB}{d+\mathrm{doffs}}.$$

`StereoSGBM.compute` restituisce valori fissi con fattore 16. Il modulo li divide una sola volta. I valori invalidi diventano NaN. Una disparità zero può descrivere una profondità finita quando doffs è positivo.

In [ ]:
# C19
# Formula visibile e verificabile, indipendente dalla funzione di supporto.
d = dense["disparity"]
den = d + cal["doffs"]
z_calcolata = np.full(d.shape, np.nan, dtype=np.float32)
valid = np.isfinite(d) & (den > 0)
z_calcolata[valid] = cal["K0"][0,0] * cal["baseline_m"] / den[valid]
assert np.allclose(z_calcolata, dense["depth"], equal_nan=True)
d_esempio = 40.0
print("Per d=40 px, Z [m] =", cal["K0"][0,0]*cal["baseline_m"]/(d_esempio+cal["doffs"]))
print("Con doffs omesso [m] =", cal["K0"][0,0]*cal["baseline_m"]/d_esempio)
mostra(gs.plot_uncertainty())

In [ ]:
# C20
# Denominatori espliciti: riferimento valutabile, predizione valida e intersezione.
M = dense["eval_mask"]
V = M & np.isfinite(d) & np.isfinite(dense["depth"])
epe = np.abs(d[V]-data["gt"][V]).mean()
print("N valutabili", int(M.sum()), "N validi", int(V.sum()), "copertura", V.sum()/M.sum())
print("EPE [px]",epe,"Bad-2 [%]",100*np.mean(np.abs(d[V]-data["gt"][V])>2))
assert np.isclose(epe, dense["metrics"]["epe_px"])
# Modifica facoltativa dopo aver salvato la baseline: blocco 3 o 7, a scelta.
# d_alt, _ = gs.stereo(data["left"],data["right"],80,block_size=7)

**C21**

**Da completare.** CP3–4. Una mappa più liscia o più piena è più accurata? Quali pixel entrano nell’EPE? Come cresce σZ con la distanza?

**Risposta unica:** includere questo checkpoint nella risposta a E03 qui sotto.

---

### E03 · Stereo guidata: errore, copertura e pixel comuni

Sulla coppia A si riusa la baseline con blocco 5 di C17 e si aggiungono soltanto i blocchi **3 e 7**. Il codice calcola disparità, profondità, tabella e un bordo già scelto, senza modificare `dense`.

1. Copiare il codice guidato in una nuova cella dopo C21. Completare **soltanto `comune_e03`**, l'intersezione delle tre maschere `validi_e03[3]`, `[5]` e `[7]`. La tabella riporta P1/P2, copertura rispetto a M, EPE e Bad2 su supporto proprio e comune.
2. Scrivere **una conclusione di 4–6 righe**, che copra anche CP3–4: quale compromesso osservi fra copertura ed errore? Perché confrontare gli stessi pixel? Quale limite noti nel bordo mostrato? Spiegare inoltre come cambia $\sigma_Z$ se la distanza raddoppia a f, B e $\sigma_d$ costanti.

Non sono richiesti altri ritagli, altre configurazioni o una relazione separata. `gs.stereo` cambia insieme **blockSize, P1=8b² e P2=32b²**: il confronto non isola il solo effetto della finestra. Le disparità sono già divise per 16; gli invalidi restano NaN. Se il supporto comune è vuoto, dichiarare il confronto non valutabile.

**Codice di partenza — una riga da completare:**

```python
# E03 — una sola riga da completare; C03, C05 e C17 già eseguite
M_e03 = gs.evaluation_mask(data)
# Si riusa la baseline b=5 e si calcolano soltanto b=3 e b=7.
configurazioni_e03 = {5: (dense["disparity"].copy(), dense["depth"].copy())}
for b_e03 in (3, 7):
    d_e03, _ = gs.stereo(data["left"], data["right"], 80, block_size=b_e03)
    Z_e03 = gs.depth_from_disparity(d_e03, cal)
    configurazioni_e03[b_e03] = (d_e03, Z_e03)
validi_e03 = {
    b: M_e03 & np.isfinite(d) & np.isfinite(Z) & (Z > 0)
    for b, (d, Z) in configurazioni_e03.items()}
comune_e03 = ...  # TODO: intersezione di validi_e03[3], [5] e [7]
if comune_e03 is Ellipsis:
    raise ValueError("Completare comune_e03 prima di eseguire.")
comune_e03 = np.asarray(comune_e03)
assert comune_e03.dtype == bool and comune_e03.shape == M_e03.shape
assert all(not np.any(comune_e03 & ~V) for V in validi_e03.values())
print("Pixel di riferimento:", int(M_e03.sum()), "Pixel comuni:", int(comune_e03.sum()))

righe_e03 = []
for b_e03 in (3, 5, 7):
    d_e03, Z_e03 = configurazioni_e03[b_e03]
    proprie_e03 = gs.dense_metrics(d_e03, Z_e03, data, M_e03)
    comuni_e03 = gs.dense_metrics(d_e03, Z_e03, data, comune_e03)
    righe_e03.append([b_e03, 8*b_e03**2, 32*b_e03**2,
        proprie_e03["n_valid"], 100*proprie_e03["coverage"],
        proprie_e03["epe_px"], proprie_e03["bad2_pct"],
        comuni_e03["epe_px"], comuni_e03["bad2_pct"]])
tabella(righe_e03, ["b", "P1", "P2", "Validi N", "Cop. su M %",
                    "EPE proprio px", "Bad2 proprio %", "EPE comune px", "Bad2 comune %"])

# Un solo bordo già scelto; non occorre progettare altri ritagli o grafici.
u0_e03, u1_e03, v0_e03, v1_e03 = 300, 600, 220, 360
fig_e03, assi_e03 = plt.subplots(1, 3, figsize=(13, 3.2), constrained_layout=True)
assi_e03[0].imshow(data["left"][v0_e03:v1_e03, u0_e03:u1_e03])
assi_e03[0].set_title("Bordo osservato — immagine sinistra")
for asse_e03, b_e03 in zip(assi_e03[1:], (3, 7)):
    d_e03 = configurazioni_e03[b_e03][0]
    im_e03 = asse_e03.imshow(np.ma.masked_invalid(d_e03[v0_e03:v1_e03, u0_e03:u1_e03]),
                               cmap="turbo", vmin=0, vmax=80)
    asse_e03.set_title(f"Disparità b={b_e03}")
fig_e03.colorbar(im_e03, ax=list(assi_e03[1:]), label="Disparità (px)")
for asse_e03 in assi_e03:
    asse_e03.set_xticks([]); asse_e03.set_yticks([])
mostra(fig_e03, "E03_bordo_stereo.png")
```

**La mia risposta E03 e al checkpoint:**

[Inserire i calcoli o la tabella richiesti e una breve interpretazione.]

**C22**

## Lab 4 · Ricostruzione sparsa a due viste
Usiamo i match di feature e stimiamo la matrice essenziale E, R e la direzione di t. Le due K normalizzano separatamente i punti. La baseline nota impone la scala una volta sola. La nuvola non deriva da un sottocampionamento della mappa densa.

Convenzione coerente con la Lezione 7: $P^{(L)}$ e $P^{(R)}$ sono le coordinate euclidee dello stesso punto 3D nei riferimenti sinistro e destro, con $P^{(R)}=R P^{(L)}+t$. Il riferimento sinistro ha $C_L=0$ e il centro destro è $C_R=-R^Tt$. Il punto omogeneo è $\widetilde P=(P^T, 1)^T$. I punti immagine in pixel sono $p_L, p_R$; le loro coordinate omogenee sono $\widetilde p_i=(u_i, v_i, 1)^T$ e i raggi normalizzati sono $x_i=K_i^{-1}\widetilde p_i$, per $i=L, R$.

Triangolazione in coordinate normalizzate con $\Pi_L^{(n)}=[I\mid 0]$ e $\Pi_R^{(n)}=[R\mid t]$. Le proiezioni in pixel sono invece $\Pi_i=K_i\Pi_i^{(n)}$, per $i=L, R$; non applicare K una seconda volta a coordinate già normalizzate.

Nelle celle del notebook i suffissi `0` e `1` corrispondono a sinistra e destra: `X` e `Xh` contengono i punti euclidei e omogenei, mentre `P0` e `P1` in C24 sono le due matrici di proiezione normalizzate. Il centro destro $C_R$ è memorizzato nel campo `sparse["C1"]`; il testo «Centro C2» negli output salvati indica lo stesso $C_R$. Filtri didattici: punti finiti, davanti a entrambe le camere, riproiezione <2 px per vista, angolo ≥1°. Il riferimento non seleziona i punti accettati.

In [ ]:
# C23
sparse = gs.sparse_lab(data, match)
sm = sparse["metrics"]
tabella([[k,sm[k]] for k in ["candidates","fit","inliers_E","finite_E","front_E","reprojection_E","accepted"]],
        ["Passaggio", "N"])
print("R:\n", sparse["R"], "\nt [m]:", sparse["t"], "\nCentro C2 [m]:", sparse["C1"])
assert np.isclose(np.linalg.norm(sparse["t"]), cal["baseline_m"])
assert np.allclose(sparse["R"]@sparse["C1"]+sparse["t"],0,atol=1e-10)
mostra(gs.plot_sparse(data, match, sparse), "04_ricostruzione_sparsa.png")
mostra(gs.plot_sparse_quality(sparse))
gs.show_metrics(sm)

In [ ]:
# C24
# Triangolazione esplicita con coordinate normalizzate e posa stimata.
p0n = gs.normalize(match["p0"],cal["K0"])
p1n = gs.normalize(match["p1"],cal["K1"])
P0 = np.column_stack([np.eye(3),np.zeros(3)])
P1 = np.column_stack([sparse["R"],sparse["t"]])
Xh = cv2.triangulatePoints(P0,P1,p0n.T,p1n.T).T
X = Xh[:,:3]/Xh[:,3,None]
assert np.allclose(X[sparse["good"]],sparse["X"][sparse["good"]])
print("Confronto profondità sugli stessi",sm["n_depth_reference"],"punti con GT:")
print("Errore relativo mediano posa stimata [%]",sm["median_relative_depth_pct"])
print("Errore relativo mediano posa nota [%]",sm["known_pose_depth_pct_same_points"])

**C25**

**Da completare.** CP5–6. La mediana di riproiezione è circa 0.091 px. Perché l’errore relativo mediano di Z può essere circa 4.20%? I filtri hanno eliminato altri punti dopo RANSAC E?

*La mia risposta:* …

**C26**

## Lab 5 · Una causa, un confronto
Prima di eseguire scrivere una previsione. Ogni studente svolge **una** challenge.

- **A:** sfocatura della sola destra, σ=2 px.
- **B:** traslazione verticale della destra +4 px, mantenendo invariato il modello usato dalla stereo.
- **C:** f×1.10 nella sola conversione disparità–profondità. Disparità, immagini e matching rimangono identici.

Registrare sia errori/copertura per ogni risultato sia gli errori sull’intersezione comune delle maschere valide. Per B il confronto esclude gli stessi bordi in entrambe le versioni.

**C27**

**Da completare.** Previsione: quale metrica cambierà, quale dovrebbe restare stabile e perché?

*La mia risposta:* …

In [ ]:
# C28
CASO = "A"  # modificare in "B" oppure "C", secondo assegnazione
exp = gs.challenge(data,dense,CASO)
mostra(gs.plot_challenge(data,exp), "05_challenge.png")
em = exp["metrics"]
rows=[]
for label,key in [("Baseline","baseline"),("Modificata","modified"),
                  ("Baseline su insieme comune","common_baseline"),("Modificata su insieme comune","common_modified")]:
    a=em[key]
    rows.append([label,a["n_evaluable"],a["n_valid"],100*a["coverage"],a["epe_px"],a["median_relative_depth_pct"]])
tabella(rows,["Risultato","N valutabili","N validi","Copertura %","EPE px","Errore Z %"])
with (OUT/"confronto_challenge.csv").open("w",newline="") as f:
    writer=csv.writer(f);writer.writerow(["Risultato","N valutabili","N validi","Copertura %","EPE px","Errore Z %"]);writer.writerows(rows)
if CASO=="C":
    assert np.array_equal(exp["modified"]["disparity"],dense["disparity"],equal_nan=True)
    assert np.allclose(exp["modified"]["depth"],1.1*dense["depth"],equal_nan=True,rtol=1e-6)

**C29**

**Da completare.** Diagnosi in 80–120 parole: causa, due evidenze numeriche, una regione dell’immagine, un intervento e un limite della conclusione.

*La mia risposta:* …

**C30**

## Consegna e discussione conclusiva
Conservare notebook eseguito, quattro figure principali, confronto CSV e diagnosi. La figura challenge è un supporto aggiuntivo.

Rispondere individualmente in cinque righe:
1. Un’evidenza che distingue coerenza da accuratezza.
2. La provenienza della scala metrica.
3. Il significato di un pixel invalido.
4. La causa dominante osservata nella challenge.
5. Un dato da migliorare nella prossima acquisizione.

**Controllo prima della consegna:** completare tutti i checkpoint e le risposte **E01–E03**, e conservare nel notebook codice, tabelle e figure delle prove aggiuntive. Le quattro figure principali già richieste restano parte della consegna. La consegna comprende un solo discussione conclusiva, quello richiesto qui; non occorre rispondere nuovamente nella scheda di consegna.

Usare una sola risposta per ciascun gruppo C06/E01, CP1/E02 e CP3–4/E03. Le altre risposte originali restano da completare.

In [ ]:
# C31
# Esportazioni: nessun file originale viene modificato.
report={"matching":match["metrics"],"rectification":rect["metrics"],"stereo":dense["metrics"],
        "sparse":sparse["metrics"],"challenge":exp["metrics"],
        "versions":{"opencv":cv2.__version__,"numpy":np.__version__},"elapsed_s":time.perf_counter()-T0}
(OUT/"metriche.json").write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
gs.export_ply(OUT/"ricostruzione_sparsa.ply", data, match, sparse)
print("Risultati salvati in",OUT)
print("Esecuzione completata.")

**C32**

## Estensione facoltativa · Audit multivista
Fuori dalle tre ore: applicare COLMAP a una breve sequenza propria con sovrapposizione e parallasse, seguendo il tutorial ufficiale. Annotare immagini registrate/totali, numero di punti, lunghezza delle tracce e riproiezione. Individuare componenti separate o immagini non registrate. Il bundle adjustment riduce il costo del modello, ma non fornisce automaticamente scala metrica o verità di riferimento. Non occorre installare COLMAP per completare questo notebook.

**C33**

## Fonti e provenienza
- Arcangelo e Cosimo Distante, *Handbook of Image Processing and Computer Vision*, vol. 3, Springer 2020: §4.6.7 pp. 383–389, §4.7.2 pp. 396–406, §6.7 pp. 585–595, §7.5.6 pp. 645–655, §7.5.7 pp. 655–665. Consultazione diretta del volume locale.
- *Visione computazionale*: terminologia e figure archiviate nella lezione 7 del corso, con crediti conservati. Il volume italiano è una fonte del corso; i grafici di questo quaderno derivano dagli esperimenti locali documentati.
- Szeliski, *Computer Vision: Algorithms and Applications*, seconda edizione, capitoli 11–12, PDF locale.
- [Middlebury 2014 e calibrazione](https://vision.middlebury.edu/stereo/data/scenes2014/), [Adirondack-perfect](https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Adirondack-perfect/). Dataset di Scharstein et al., GCPR 2014.
- [OpenCV calib3d](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html), [StereoSGBM](https://docs.opencv.org/4.x/d2/d85/classcv_1_1StereoSGBM.html), [matching](https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html).
- [COLMAP, tutorial ufficiale](https://colmap.github.io/tutorial.html).

Consultazione web e acquisizione dati: 12 settembre 2026. I grafici sono risultati riproducibili di questo laboratorio. Nessuna metrica viene presentata come risultato ufficiale del benchmark.